# 29.07 - Class-imbalance training: weighted sampling

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Sampler training comparison and a strategy note covering standard sampling, weighted loss, and `WeightedRandomSampler`.

You will inspect the distribution produced by a weighted sampler and compare three controlled training strategies on a noisy imbalanced image task.


## Core Ideas

- `WeightedRandomSampler` consumes one weight per sample, not one weight per class.
- Convert class weights to sample weights with `class_weights[train_labels]`.
- When a sampler is supplied, do not also set `shuffle=True`.
- `replacement=True` lets minority samples appear repeatedly; this changes the sampled training distribution and can overfit rare examples.
- Weighted loss keeps the observed batches natural but changes each example's loss contribution.
- Keep the number of sampled observations per epoch equal across strategies when the goal is to isolate the effect of the sampling distribution.
- A tiny minority validation support makes recall jump in coarse steps. Use the full prepared split, print class support, and treat one synthetic run as evidence rather than a universal ranking.
- Do not combine sampling and weighted loss blindly. Compare them independently first.


In [69]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score

SEED = 29
NUM_CLASSES = 3
np.random.seed(SEED)
torch.manual_seed(SEED)


## Prepared Harder Imbalanced Image Data

The fixture uses shifted structures, distractors, occlusion, and severe class imbalance. The default contains 360 images so the untouched 25% validation split has 90 observations, including 12 rare-class examples. That remains fast on CPU while making per-class changes much easier to interpret than a three-example rare-class validation set.

**Return structure — `make_imbalanced_pattern_images`:** Returns a tuple. Position 0 is a CPU `torch.float32` tensor `[N,3,24,24]` with values in `[0,1]`. Position 1 is a CPU `torch.long` tensor `[N]`. `N=sum(counts)` and labels are contiguous from zero.


In [70]:
def make_imbalanced_pattern_images(counts=(240, 72, 48), image_size=24, seed=29):
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for class_index, class_count in enumerate(counts):
        for sample_index in range(class_count):
            image = rng.normal(0.18, 0.22, size=(3, image_size, image_size)).astype(np.float32)
            shift = int(rng.integers(-3, 4))
            main_channel = int(rng.integers(0, 3))
            detail_channel = (main_channel + int(rng.integers(1, 3))) % 3
            if class_index in (0, 2):
                center = image_size // 2 + shift
                image[main_channel, 3:image_size - 3, center - 2:center + 2] += 0.52
                if class_index == 2:
                    # The rare class shares the majority vertical bar and differs
                    # only by a short, partially noisy cross-piece.
                    detail_row = image_size // 2 + int(rng.integers(-3, 4))
                    image[detail_channel, detail_row - 1:detail_row + 2, center - 6:center + 7] += 0.46
            elif class_index == 1:
                center = image_size // 2 + shift
                image[main_channel, center - 2:center + 2, 3:image_size - 3] += 0.52
            # Every class receives an unrelated patch and occasional occlusion.
            patch_top = int(rng.integers(2, image_size - 6))
            patch_left = int(rng.integers(2, image_size - 6))
            image[detail_channel, patch_top:patch_top + 4, patch_left:patch_left + 4] += 0.30
            if sample_index % 3 == 0:
                top = int(rng.integers(4, image_size - 8))
                left = int(rng.integers(4, image_size - 8))
                image[:, top:top + 5, left:left + 6] *= 0.08
            images.append(np.clip(image, 0.0, 1.0))
            labels.append(class_index)
    order = rng.permutation(len(labels))
    return torch.tensor(np.stack(images)[order], dtype=torch.float32), torch.tensor(np.asarray(labels)[order], dtype=torch.long)


images, labels = make_imbalanced_pattern_images()
print("dataset:", images.shape, images.dtype, torch.bincount(labels).tolist())


dataset: torch.Size([360, 3, 24, 24]) torch.float32 [240, 72, 48]


## Prepared Split and Model Helpers

These are supplied so the exercises remain focused on sampling strategy.

**Return structure — `prepare_day29_split`:** Returns a `dict` containing CPU tensors `train_images`, `val_images` (`torch.float32` rank 4), `train_labels`, `val_labels`, `train_indices`, and `val_indices` (`torch.long` rank 1). The split is stratified, disjoint, and normalized using training statistics only.


In [71]:
def prepare_day29_split(images, labels, val_fraction=0.25, seed=SEED):
    indices = np.arange(len(labels))
    train_array, val_array = train_test_split(
        indices,
        test_size=val_fraction,
        random_state=seed,
        stratify=labels.numpy(),
    )
    train_indices = torch.tensor(train_array, dtype=torch.long)
    val_indices = torch.tensor(val_array, dtype=torch.long)
    raw_train = images[train_indices]
    raw_val = images[val_indices]
    mean = raw_train.mean()
    std = raw_train.std().clamp_min(1e-6)
    return {
        "train_images": ((raw_train - mean) / std).float(),
        "val_images": ((raw_val - mean) / std).float(),
        "train_labels": labels[train_indices].long(),
        "val_labels": labels[val_indices].long(),
        "train_indices": train_indices,
        "val_indices": val_indices,
    }


split = prepare_day29_split(images, labels)
print("train/validation counts:", torch.bincount(split["train_labels"]).tolist(), torch.bincount(split["val_labels"]).tolist())


train/validation counts: [180, 54, 36] [60, 18, 12]


**Return structure — `TinySamplerCNN`:** A callable `nn.Module`. Construction returns a CPU module. Calling it with a CPU `torch.float32` tensor `[N,3,H,W]` returns CPU `torch.float32` logits `[N,num_classes]`.


In [72]:
class TinySamplerCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 12, 3, padding=1),
            nn.BatchNorm2d(12),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(12, 24, 3, padding=1),
            nn.BatchNorm2d(24),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(24, num_classes)

    def forward(self, batch):
        return self.classifier(self.features(batch).flatten(1))


print("prepared model output:", TinySamplerCNN(NUM_CLASSES)(split["train_images"][:3]).shape)


prepared model output: torch.Size([3, 3])


## Exercise 29-A: Build a weighted sampler

Calculate inverse-frequency class weights, map them to each training observation, and sample with replacement. Set `num_samples=N_train` so standard, weighted-loss, and weighted-sampler runs receive the same number of training observations per epoch; the controlled difference is the sampled class distribution.

**Return structure — `build_weighted_sampler`:** Returns a tuple. Position 0 is a `WeightedRandomSampler` with `replacement=True` and `num_samples=N_train`. Position 1 is a CPU `torch.float64` tensor `[N_train]` containing one positive finite weight per training observation.


In [73]:
# TODO 29-A

def build_weighted_sampler(train_labels, num_classes, seed=SEED):
    counts = torch.bincount(train_labels,minlength = num_classes).double()
    if torch.any(counts == 0) :
        raise ValueError()
    class_weights = counts.reciprocal()
    sample_weights = class_weights[train_labels]
    epoch_size = len(train_labels)
    sampler = WeightedRandomSampler(
        weights = sample_weights,
        num_samples = epoch_size,
        replacement = True,
        generator = torch.Generator().manual_seed(seed)
    )
    return (sampler,sample_weights)


# Smoke check: run this after implementing the function above.
smoke_sampler, smoke_sample_weights = build_weighted_sampler(split["train_labels"], NUM_CLASSES)
print("sampler epoch size:", smoke_sampler.num_samples, "sample weights:", smoke_sample_weights.shape)


sampler epoch size: 270 sample weights: torch.Size([270])


## Exercise 29-B: Audit the sampled distribution

Draw one sampler epoch and compare natural and sampled counts.

**Return structure — `audit_sampled_epoch`:** Returns a `dict` with `natural_counts` and `sampled_counts`, CPU `torch.long` tensors `[C]`; `sampled_indices`, a CPU `torch.long` tensor `[sampler.num_samples]`; and `max_min_ratio`, a finite positive Python `float` equal to maximum sampled count divided by minimum sampled count.


In [74]:
# TODO 29-B
def audit_sampled_epoch(train_labels, sampler, num_classes):
    sampled_indices = torch.tensor(list(sampler))
    natural_counts = torch.bincount(train_labels,minlength=num_classes)
    sampled_counts = torch.bincount(train_labels[sampled_indices],minlength=num_classes)
    max_min_ratio = sampled_counts.max().item() / sampled_counts.min().item()
    return {
        "natural_counts" : natural_counts,
        "sampled_counts" : sampled_counts,
        "sampled_indices" : sampled_indices,
        "max_min_ratio" : max_min_ratio
    }


# Smoke check: run this after implementing the function above.
sampling_audit = audit_sampled_epoch(split["train_labels"], smoke_sampler, NUM_CLASSES)
print("natural/sampled:", sampling_audit["natural_counts"].tolist(), sampling_audit["sampled_counts"].tolist())


natural/sampled: [180, 54, 36] [77, 106, 87]


## Exercise 29-C: Train one imbalance strategy

Support exactly `standard`, `weighted_loss`, and `weighted_sampler`. Use only one intervention at a time and evaluate every validation observation.

**Return structure — `train_imbalance_strategy`:** Returns a `dict` with `strategy` (`str`), `history` (`list[float]` of length `epochs`), and `metrics`. `metrics` contains `accuracy`, `macro_f1` (Python floats), `per_class_recall` (CPU `torch.float32` tensor `[C]`), `per_class_support` (CPU `torch.long` tensor `[C]` counting every validation label), and `confusion_matrix` (CPU `torch.long` tensor `[C,C]`).


In [100]:
# TODO 29-C
def train_imbalance_strategy(split, num_classes, strategy, epochs=6, seed=SEED):
    torch.manual_seed(seed)
    train_images = split["train_images"]
    train_labels = split["train_labels"]
    criterion = nn.CrossEntropyLoss()
    sampler = None
    if strategy == "weighted_loss" : 
        counts = torch.bincount(split["train_labels"],minlength = num_classes).float()
        weights = counts.sum() / (num_classes * counts)
        criterion = nn.CrossEntropyLoss(weight = weights)
    elif strategy == "weighted_sampler" :
        sampler = build_weighted_sampler(split["train_labels"],num_classes,seed = seed)[0]
    model = TinySamplerCNN(num_classes)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr = 0.005,
        weight_decay = 0.01
    )
    if sampler is not None : 
        train_loader = DataLoader(TensorDataset(train_images,train_labels), batch_size = 18, sampler = sampler)
    else : 
        train_loader = DataLoader(TensorDataset(train_images,train_labels), batch_size = 18, shuffle = True, generator = torch.Generator().manual_seed(seed))
    val_loader = DataLoader(TensorDataset(split["val_images"],split["val_labels"]), batch_size = 18, shuffle = False)
    model.train()
    history = []
    for _ in range(epochs) :
        total_loss = 0.0
        total_samples = 0
        for images, labels in train_loader :
            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits,labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(labels)
            total_samples += len(labels)

        history.append(total_loss/total_samples)
    all_preds = []
    all_labels = []
    model.eval()
    with torch.inference_mode() : 
        for images, labels in val_loader : 
            logits = model(images)
            preds = logits.argmax(dim = 1)
            all_preds.append(preds)
            all_labels.append(labels)
        all_preds = torch.cat(all_preds, dim = 0)
        all_labels = torch.cat(all_labels, dim = 0)
        per_class_support = torch.bincount(all_labels,minlength = num_classes)
    all_preds = all_preds.numpy()
    all_labels = all_labels.numpy()
    accuracy = accuracy_score(all_labels,all_preds)
    macro_f1 = f1_score(all_labels,all_preds,average = "macro")
    per_class_recall = recall_score(all_labels,all_preds,average = None)
    per_class_recall = torch.from_numpy(per_class_recall).to(torch.float32)
    confusion__matrix = confusion_matrix(all_labels,all_preds)
    confusion__matrix = torch.from_numpy(confusion__matrix)
    return {
        "strategy" : strategy,
        "history" : history,
        "metrics" : {
            "accuracy" : accuracy,
            "macro_f1" : macro_f1,
            "per_class_recall" : per_class_recall,
            "confusion_matrix" : confusion__matrix,
            "per_class_support" : per_class_support
        }
    }




# Smoke check: run this after implementing the function above.
smoke_strategy = train_imbalance_strategy(split, NUM_CLASSES, "standard", epochs=1)
print("standard smoke Macro-F1:", smoke_strategy["metrics"]["macro_f1"])


standard smoke Macro-F1: 0.26666666666666666


## Exercise 29-D: Compare all three strategies

Run controlled experiments on the same complete training and validation split. Keep initialization, epoch count, and observations per epoch aligned, then report full-validation metrics without assuming that the most aggressive balancing method must win.

**Return structure — `compare_imbalance_strategies`:** Returns a `dict` with exactly `standard`, `weighted_loss`, and `weighted_sampler`. Each value is the matching dictionary from `train_imbalance_strategy` and uses the same requested epoch count.


In [108]:
# TODO 29-D
def compare_imbalance_strategies(split, num_classes, epochs=6, seed=SEED):
    return {
        "standard" : train_imbalance_strategy(split, num_classes, "standard", epochs, seed),
        "weighted_loss" : train_imbalance_strategy(split, num_classes, "weighted_loss", epochs, seed),
        "weighted_sampler" : train_imbalance_strategy(split, num_classes, "weighted_sampler", epochs, seed)
    }



# Smoke check: this is the decision experiment, so use the complete prepared split.
strategy_results = compare_imbalance_strategies(split, NUM_CLASSES, epochs=23)
baseline_macro_f1 = strategy_results["standard"]["metrics"]["macro_f1"]
print("full split:", len(split["train_labels"]), "train /", len(split["val_labels"]), "validation")
print("validation support:", torch.bincount(split["val_labels"], minlength=NUM_CLASSES).tolist())
print(f"{'strategy':<18} {'accuracy':>9} {'macro_f1':>10} {'delta':>9}  per-class recall")
for name, result in strategy_results.items():
    metrics = result["metrics"]
    delta = metrics["macro_f1"] - baseline_macro_f1
    recalls = [round(value, 3) for value in metrics["per_class_recall"].tolist()]
    print(f"{name:<18} {metrics['accuracy']:>9.3f} {metrics['macro_f1']:>10.3f} {delta:>+9.3f}  {recalls}")


full split: 270 train / 90 validation
validation support: [60, 18, 12]
strategy            accuracy   macro_f1     delta  per-class recall
standard               0.833      0.641    +0.000  [0.983, 0.833, 0.083]
weighted_loss          0.867      0.821    +0.180  [0.883, 0.944, 0.667]
weighted_sampler       0.922      0.874    +0.233  [1.0, 0.889, 0.583]


## Test Cases

Run this cell after completing all TODO cells. It checks contracts and verifies that the decision experiment used the complete prepared train/validation split. It also reprints the interpretable comparison rather than reducing the outcome to a pass/fail signal. A correct implementation ends by printing `Day 29 tests passed`.

**Return structure — `run_day29_tests`:** Returns `None`. Success is communicated by assertions completing, a full-split evidence table, and the exact final printed message `Day 29 tests passed`.


In [89]:
def run_day29_tests():
    dataset_counts = torch.bincount(labels, minlength=NUM_CLASSES)
    assert dataset_counts.tolist() == [240, 72, 48]
    assert len(split["train_labels"]) + len(split["val_labels"]) == len(labels)
    assert set(split["train_indices"].tolist()).isdisjoint(split["val_indices"].tolist())
    validation_support = torch.bincount(split["val_labels"], minlength=NUM_CLASSES)
    assert validation_support.tolist() == [60, 18, 12]
    assert int(validation_support.min()) >= 10

    sampler, sample_weights = build_weighted_sampler(split["train_labels"], NUM_CLASSES, seed=SEED)
    counts = torch.bincount(split["train_labels"], minlength=NUM_CLASSES)
    assert isinstance(sampler, WeightedRandomSampler) and sampler.replacement is True
    assert sampler.num_samples == len(split["train_labels"])
    assert sample_weights.shape == split["train_labels"].shape and sample_weights.dtype == torch.float64
    assert torch.allclose(sample_weights, counts.double().reciprocal()[split["train_labels"]])

    audit_sampler, _ = build_weighted_sampler(split["train_labels"], NUM_CLASSES, seed=SEED)
    audit = audit_sampled_epoch(split["train_labels"], audit_sampler, NUM_CLASSES)
    assert set(audit) == {"natural_counts", "sampled_counts", "sampled_indices", "max_min_ratio"}
    assert audit["sampled_indices"].shape == (sampler.num_samples,)
    assert int(audit["sampled_counts"].sum()) == sampler.num_samples
    assert audit["max_min_ratio"] < 2.0

    assert set(strategy_results) == {"standard", "weighted_loss", "weighted_sampler"}
    expected_metric_keys = {"accuracy", "macro_f1", "per_class_recall", "per_class_support", "confusion_matrix"}
    for name, result in strategy_results.items():
        assert result["strategy"] == name and len(result["history"]) == 6
        metrics = result["metrics"]
        assert set(metrics) == expected_metric_keys
        assert metrics["per_class_recall"].shape == (NUM_CLASSES,)
        assert torch.equal(metrics["per_class_support"], validation_support)
        assert metrics["confusion_matrix"].shape == (NUM_CLASSES, NUM_CLASSES)
        assert int(metrics["confusion_matrix"].sum()) == len(split["val_labels"])

    baseline = strategy_results["standard"]["metrics"]["macro_f1"]
    print("\nVerified full-validation decision evidence:")
    for name, result in strategy_results.items():
        metrics = result["metrics"]
        delta = metrics["macro_f1"] - baseline
        print(name, "Macro-F1:", round(metrics["macro_f1"], 3), "delta:", round(delta, 3),
              "recall:", [round(value, 3) for value in metrics["per_class_recall"].tolist()])
    print("Day 29 tests passed")


run_day29_tests()



Verified full-validation decision evidence:
standard Macro-F1: 0.814 delta: 0.0 recall: [0.833, 1.0, 0.667]
weighted_loss Macro-F1: 0.826 delta: 0.012 recall: [0.9, 0.833, 0.833]
weighted_sampler Macro-F1: 0.615 delta: -0.199 recall: [0.867, 0.222, 1.0]
Day 29 tests passed


## Day 29 Checklist

- [ ] I converted class weights into per-sample weights.
- [ ] I used a sampler instead of `shuffle=True`.
- [ ] I kept observations per epoch equal across controlled strategy runs.
- [ ] I inspected the sampled class distribution.
- [ ] I checked validation class support and evaluated every validation observation.
- [ ] I compared accuracy, Macro-F1, per-class recall, and deltas from the baseline.
- [ ] I compared sampling and weighted loss independently.
- [ ] I can explain replacement, epoch size, minority overfitting, and when to prefer each strategy.
